In [ ]:
# 02_features.ipynb -- Study 2 feature table: labels, lag/rolling slot features,
# corridor/cross-border broadcast, Study 1 residual signal, class balance
# !pip install lightgbm -q

import pandas as pd

import features as f

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)

# build_feature_table() with no study1_residual argument computes it itself, by
# backtesting Study 1's baseline (train <=2022, val 2023, predict 2024+) --
# see features.py's build_study1_residual_signal() docstring for why this is a
# backtest approximation rather than a reproduction of predict.py's live daily output.
feat_df = f.build_feature_table(scada)

print("feature table shape:", feat_df.shape)
print("columns:", feat_df.columns.tolist())

print("\nviolation_lead: ", feat_df["violation_lead"].notna().sum(), "labeled rows, rate =",
      round(feat_df["violation_lead"].mean(), 4))
print("ramp_lead:      ", feat_df["ramp_lead"].notna().sum(), "labeled rows, rate =",
      round(feat_df["ramp_lead"].mean(), 4))

print("\nrows with unresolvable lead label (dropped before training):")
print(" violation_lead NaN:", feat_df["violation_lead"].isna().sum())
print(" ramp_lead NaN:", feat_df["ramp_lead"].isna().sum())

print("\nnull rate per feature column:")
print(feat_df[f.FEATURE_COLS].isna().mean().round(4).sort_values(ascending=False))

# --- Notes ---
# violation_lead and ramp_lead are both "did the event happen in any of the next 1-4
# slots (15-60 min)" -- OR'd over the lookahead window, so their positive rate is
# necessarily higher than the raw per-slot event rate from 01_eda.ipynb (0.89% ->
# ~2.4% for violation, 6.1% -> ~15% for ramp, since the OR captures any of 4 chances).
#
# study1_residual_mw nulls are concentrated at the very start (before Study 1's 2024
# backtest window begins) and the very end (today's row, whose "tomorrow" hasn't
# happened yet in study1_daily.csv) -- structural, not a data quality bug.
#
# LightGBM handles NaN features natively (treated as a learnable split direction), so
# no imputation is applied here -- imputing would risk inventing signal that isn't
# really there, especially study1_residual_mw, whose nulls are
# meaningfully informative (e.g. "no residual yet" vs "residual was exactly 0").
#
# share_res_pct and the 11 corridor/cross-border columns are computed here (they're
# still part of feat_df) but deliberately excluded from FEATURE_COLS -- see
# features.py's DAILY_BROADCAST_COLS comment: they are whole-day aggregates broadcast
# identically to all 96 slots of a day, and removing them from the classifiers'
# feature set improved both targets rather than hurting them (found 2026-07-11).
